# OVRO-LWA HEALPix nested-tile detect (Option 2)

Experiment: coadd hourly FITS onto a **HEALPix** map (`nside_map=2048`), write MAP+WEIGHT FITS,
project **nested** tiles (`nside_tile=4`, TAN, overlap 0.2) via `reproject_from_healpix`, then
`run_pybdsf_on_hdu` on each tile.

This is a sibling to `ovro_lwa_mosaic_detect.ipynb` (planar SIN coadd). It does **not** replace
the per-hour → LST-merge catalog pipeline.

**Notes**
- Do **not** call `blank_below_elevation` on tile HDUs (CRVAL is tile center, not zenith).
- Elevation blanking happens inside `coadd_fits(..., min_elevation=...)` on native hourly WCS.
- Catalogs are tagged with `tile_ipix` / `nside_tile`. Overlap ≠ edge merge (out of scope).
- Requires `lwa-catalog[analyze,detect]` and a current editable `lwa-healpix`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from astropy.io import fits

from lwa_catalog.create.detect import DEFAULT_BDSF_KW
from lwa_catalog.create.discover import discover_fits_files, discovered_slots
from lwa_catalog.create.healpix_detect import (
    detect_sources_on_healpix_tiles,
    median_beam_from_paths,
    restfreq_hz_from_header,
)
from lwa_catalog.io import read_lst_merged, write_table
from lwa_catalog.paths import CatalogLayout
from lwa_healpix import coadd_fits, read_healpix_fits, write_healpix_fits

# --- paths (edit for your machine) ---
FITS_ROOT = Path("/lustre/pipeline/images")  # operator: set me
FITS_GLOB = "??h_Full/*_I_deep_Taper_Robust+0.0_dewarped*fits"
BAND = "Full"
CATALOG_DIR = Path("/fast/claw/catalogs/metacatalog_coaddR-0.75")  # optional compare
OUTPUT_DIR = Path("/fast/claw/healpix_tile_detect")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HEALPIX_FITS = OUTPUT_DIR / f"healpix_{BAND}_nside2048.fits"
TILE_CATALOG = OUTPUT_DIR / f"sources_healpix_tiles_{BAND}.parquet"
REUSE_HEALPIX_FITS = True

NSIDE_MAP = 2048
NSIDE_TILE = 4
OVERLAP = 0.2
MIN_ELEVATION_DEG = 10.0
COORD_FRAME = "equatorial"
NESTED = True

BDSF_KW = dict(DEFAULT_BDSF_KW)
# BDSF_KW["thresh_isl"] = 3.0  # optional override


In [ ]:
fits_files = discover_fits_files(FITS_ROOT, patterns=(FITS_GLOB,))
slots = discovered_slots(fits_files)
metas = [m for m in fits_files if m.band == BAND]
# one path per LST hour (first match)
by_lst = {}
for m in metas:
    by_lst.setdefault(m.lst_hour, m)
paths = [by_lst[h].path for h in sorted(by_lst)]
print(f"{BAND}: {len(paths)} LST hours from {len(metas)} discovered files")
print("slots sample:", list(slots)[:5], "...")


In [ ]:
if REUSE_HEALPIX_FITS and HEALPIX_FITS.is_file():
    healpix_map, weight, meta = read_healpix_fits(HEALPIX_FITS)
    print(f"Reused {HEALPIX_FITS}: nside={meta['nside']} nested={meta['nested']} frame={meta['coord_frame']}")
else:
    healpix_map, weight = coadd_fits(
        paths,
        nside=NSIDE_MAP,
        nested=NESTED,
        coord_frame=COORD_FRAME,
        min_elevation=MIN_ELEVATION_DEG,
    )
    write_healpix_fits(
        HEALPIX_FITS,
        healpix_map,
        weight,
        nside=NSIDE_MAP,
        nested=NESTED,
        coord_frame=COORD_FRAME,
        overwrite=True,
    )
    print(f"Wrote {HEALPIX_FITS}  weight>0={(np.asarray(weight) > 0).sum()} / {weight.size}")

bmaj, bmin, bpa = median_beam_from_paths(paths)
restfreq = restfreq_hz_from_header(fits.getheader(paths[0]))
print(f"median beam BMAJ={bmaj:.5f} BMIN={bmin:.5f} BPA={bpa:.3f} deg  RESTFREQ={restfreq:.3e} Hz")


In [ ]:
catalog = detect_sources_on_healpix_tiles(
    healpix_map,
    weight,
    nside_map=NSIDE_MAP,
    nside_tile=NSIDE_TILE,
    overlap=OVERLAP,
    coord_frame=COORD_FRAME,
    nested=NESTED,
    bmaj=bmaj,
    bmin=bmin,
    bpa=bpa,
    restfreq_hz=restfreq,
    band=BAND,
    bdsf_kw=BDSF_KW,
    skip_empty=True,
)
write_table(catalog, TILE_CATALOG)
print(f"Wrote {TILE_CATALOG}: {len(catalog)} rows, {catalog['tile_ipix'].nunique() if len(catalog) else 0} tiles with sources")
catalog.head()


## Optional: compare to LST-merged catalog

Northern-sky peek only — not a scientific gate.


In [ ]:
if CATALOG_DIR.is_dir() and len(catalog):
    layout = CatalogLayout(CATALOG_DIR)
    lst = read_lst_merged(layout, BAND)
    north = catalog[catalog["DEC"] > 0]
    print(f"tile catalog north: {len(north)}  lst-merged: {len(lst)}")
else:
    print("Skip compare (missing CATALOG_DIR or empty tile catalog)")
